# PINNs on Heat equation

## Problem

- Domain: $(x, t) \in \Omega = (0,1) \times (0,1)$
- Parabolic Boundary: 
    - Initial: $\partial_i \Omega = (0,1) \times \{0\}$
    - State Boundary: $\partial_0 \Omega = \{0\} \times (0,1)$ and $\partial_1 \Omega = \{1\} \times (0,1)$
- Equation: 
    $$ u_t = u_{xx} \hbox{ on } \Omega.$$
- Data: 
    - Initial data: $u(x, t) = \phi(x)$ on $\partial_i \Omega$
    - Boundary data: $u(x, t) = 0$ on $\partial_0 \Omega$ and $\partial_1 \Omega$.

In [2]:
# code starts here
import torch

import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# Reproducibility
torch.manual_seed(42)

__Example__
When the initial data $\phi(x) = 1$, the explicit solution shall be
$$ u(x, t) = \sum_{n \in 2 \mathbb N -1} \frac{4}{n\pi} \sin (n \pi x) e^{-n^2 \pi^2 t}.$$

In [10]:
class Heat:
    def __init__(
        self,
        rec_domain=None,  # Rectangular domain: [[x0, t0], [x1, t1]]
        initial_condition=lambda x: torch.ones_like(x),
        boundary_condition0=lambda t: torch.zeros_like(t),
        boundary_condition1=lambda t: torch.zeros_like(t),
    ):
        if rec_domain is None:
            rec_domain = [[0.0, 0.0], [1.0, 1.0]] # [x0, t0], [x1, t1]]
        self.domain = rec_domain
        self.initial_condition = initial_condition
        self.boundary_condition0 = boundary_condition0
        self.boundary_condition1 = boundary_condition1

    # PINNs model
    def pinns_model(self):
        return nn.Sequential(
            nn.Linear(2, 20),  # Input: (x, t)
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 1),  # Output: u(x, t)
        )

    # Loss function for PDE residual: u_t - u_xx
    def loss_function(self, model, x):  # x: (N, 2) tensor of (x, t)
        x = x.clone().detach().requires_grad_(True)
        u = model(x)

        grads = torch.autograd.grad(
            outputs=u,
            inputs=x,
            grad_outputs=torch.ones_like(u),
            create_graph=True,
        )[0]
        u_x = grads[:, 0:1]
        u_t = grads[:, 1:2]

        u_xx = torch.autograd.grad(
            outputs=u_x,
            inputs=x,
            grad_outputs=torch.ones_like(u_x),
            create_graph=True,
        )[0][:, 0:1]

        residual = u_t - u_xx
        return torch.mean(residual**2)

    # Loss function for initial and boundary conditions
    def loss_ic_bc(self, model, x_ic, x_bc0, x_bc1): # x_ic: (N_ic, 2), x_bc0: (N_bc, 2), x_bc1: (N_bc, 2)      
        # Initial condition loss
        u_ic_pred = model(x_ic)
        u_ic_true = self.initial_condition(x_ic[:, 0:1])
        loss_ic = torch.mean((u_ic_pred - u_ic_true)**2)

        # Boundary condition loss
        u_bc0_pred = model(x_bc0)
        u_bc0_true = self.boundary_condition0(x_bc0[:, 1:2])
        loss_bc0 = torch.mean((u_bc0_pred - u_bc0_true)**2)

        u_bc1_pred = model(x_bc1)
        u_bc1_true = self.boundary_condition1(x_bc1[:, 1:2])
        loss_bc1 = torch.mean((u_bc1_pred - u_bc1_true)**2)

        return loss_ic + loss_bc0 + loss_bc1

    # Training loop
    def train(self, model, epochs=1000, lr=1e-3,
              n_ic=100, n_bc=100, n_colloc=1000):
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        device = next(model.parameters()).device
        domain = torch.tensor(self.domain, dtype=torch.float32, device=device)
        x_min = domain[0][0]
        t_min = domain[0][1]
        x_max = domain[1][0]
        t_max = domain[1][1]

        # Sample points for initial and boundary conditions
        x_ic = torch.rand(n_ic, 1, device=device) * (x_max - x_min) + x_min
        x_ic = torch.stack([x_ic.squeeze(), torch.tensor(t_min, device=device).expand_as(x_ic)], dim=1)
